## ပြင်ဆင်ထားသောအပိုင်းများ / Change labels

Percentage မပါသော ပထမ notebook နှင့် နှိုင်းယှဉ်ထားသည်။ ဒီဖိုင်မှာ ရှိပြီးသား percentage version ကို label များသာ ထပ်တပ်ထားသည်။

| Label | နေရာ | ပြောင်းလဲထားသည်များ |
|---|---|---|
| 🟩 NEW | 7A | အဖြေစစ်ပြီး verdict ဖြည့်ရန် cell အသစ် |
| 🟩 NEW | 7B | Percentage တွက်ချက်မှု၊ result display နှင့် summary files |
| 🟨 UPDATED | 8 | Percentage summaries နှင့် reviewed results ကို ZIP ထဲထည့်ရန် |
| ⬜ UNCHANGED | 1–7 | မူလ setup၊ model loading၊ test မေးခွန်းများနှင့် inference result ထုတ်ခြင်း မပြောင်းပါ |

**အသုံးပြုပုံ:** လက်ရှိ session မှာ အဆင့် ၇ ပြီးပြီး `results` ရှိနေပါက 7A → 7B → 8 ကိုသာ ဆက် run ပါ။ Label များကြည့်ရန် Runtime ကို restart လုပ်ရန် မလိုပါ။ Notebook အသစ်ဖွင့်ရုံနဲ့ ယခင် session ရဲ့ `results` အလိုအလျောက် မရပါ — လက်ရှိ notebook ထဲသို့ သက်ဆိုင်ရာ cells ကို copy လုပ်ပါ။


# DeepSeek-R1-Distill-Llama-8B: Colab T4 evaluation
Run cells in order. This notebook evaluates 4-bit NF4 inference, not the original full-precision weights.
No model outputs are pre-filled. GPU execution must happen in your Colab runtime.
Use public/synthetic prompts only, not confidential company data.

## 1. Install
Use a fresh Colab GPU runtime. Keep the bundled PyTorch. After this cell finishes, use **Runtime → Restart session**, then continue at cell 2 (do not repeat the install).


In [6]:
%pip install --upgrade transformers accelerate bitsandbytes safetensors


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 142.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 182.0 MB/s  0:00:00m0:00:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:━━━━━━━━━━━ 0/3 [transformers]
      Successfully uninstalled transformers-5.20.20m 0/3 [transformers]
  Attempting uninstall: acceleratem━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [bitsandbytes]
    Found existing installation: accelerate 1.15.0━━━━━━━━━━━━ 1/3 [bitsandbytes]
    Uninstalling accelerate-1.15.0:m━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [bitsandbytes]
      Successfully uninstalled accelerate-1.15.0━━━━━━━━━━━━━━ 1/3 [bitsandbytes]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [accelerate]3 [accelerate]
Note: you may need to restart the kernel to use updated packages.


## 2. Check the actual runtime
This runs a real CUDA operation. Stop here if it fails; do not blindly downgrade PyTorch.


In [7]:
import sys
import json
import time
import csv
from pathlib import Path
from importlib.metadata import version
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
for package in ["transformers", "accelerate", "bitsandbytes", "safetensors"]:
    print(package, version(package))
assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
x = torch.ones((16, 16), device="cuda", dtype=torch.float16)
y = x @ x
assert y[0, 0].item() == 16
print("CUDA FP16 matrix multiplication: OK")
del x, y


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130 CUDA build: 13.0
transformers 5.23.0
accelerate 1.16.0
bitsandbytes 0.50.2
safetensors 0.8.0
GPU: Tesla T4
Compute capability: (7, 5)
VRAM GiB: 14.56
CUDA FP16 matrix multiplication: OK


## 3. Load the model once
T4 uses FP16, not BF16. Do not install FlashAttention-2. SDPA can select a supported backend.
The original checkpoint download is larger than its quantized runtime footprint. First load may take time.
The public-model unauthenticated warning alone is not a failure. Optional HF_TOKEN belongs in Colab Secrets, never in shared code.


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    device_map={"": 0},
    dtype=torch.float16,
    attn_implementation="sdpa",
)
model.eval()
print("Model loaded; inference is not yet verified.")
print("Allocated VRAM GiB:", round(torch.cuda.memory_allocated() / 2**30, 2))


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded; inference is not yet verified.
Allocated VRAM GiB: 5.32


## 4. Inference helper
Preserves raw output and reports final-answer extraction separately. Missing `</think>` means manual review, not an automatic wrong answer.
Token/s here includes reasoning tokens and prompt processing time; it is not time-to-first-token.


In [11]:
def extract_final(raw):
    if "</think>" in raw:
        return raw.rsplit("</think>", 1)[1].strip(), "after_think_boundary"
    return None, "missing_think_boundary_review_raw"


def run_prompt(prompt, max_new_tokens=1024, seed=42):
    set_seed(seed)
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    if not rendered.rstrip().endswith("<think>"):
        rendered += "<think>\n"
    inputs = tokenizer(rendered, add_special_tokens=False, return_tensors="pt").to("cuda:0")
    input_tokens = inputs["input_ids"].shape[-1]
    if input_tokens > 2048:
        raise ValueError("This starter notebook limits inputs to 2048 tokens; shorten the prompt.")
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=model.generation_config.eos_token_id,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    generated = output[0, input_tokens:]
    raw = tokenizer.decode(generated, skip_special_tokens=True)
    final, extraction = extract_final(raw)
    count = generated.numel()
    eos = model.generation_config.eos_token_id
    eos_ids = {eos} if isinstance(eos, int) else set(eos or [])
    ended_with_eos = count > 0 and generated[-1].item() in eos_ids
    return {
        "prompt": prompt,
        "raw_output": raw,
        "final_answer": final,
        "answer_extraction": extraction,
        "input_tokens": input_tokens,
        "output_tokens": count,
        "seconds": round(elapsed, 3),
        "generated_tokens_per_second": round(count / elapsed, 3),
        "peak_allocated_vram_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3),
        "hit_token_limit": count >= max_new_tokens and not ended_with_eos,
        "seed": seed,
        "max_new_tokens": max_new_tokens,
    }


## 5. Smoke test: actual quantized inference
If this cell generates an answer, quantized inference works. If the token limit is hit, the answer may be unfinished.


In [12]:
smoke = run_prompt("What is the capital of Japan? Give a short final answer.", max_new_tokens=512)
print("RAW OUTPUT:\n", smoke["raw_output"])
print("\nFINAL:", smoke["final_answer"])
print("Token limit hit:", smoke["hit_token_limit"])
print("Seconds:", smoke["seconds"])


RAW OUTPUT:
 Okay, so I need to figure out the capital of Japan. Hmm, I remember from school that Japan's capital is a city, but I'm not exactly sure which one. Let me think. I've heard of Tokyo before; it's a big city, right? I think it's in the east part of Japan, near the coast. Wait, is it the largest city in Japan? Yeah, I believe that's correct. I also recall hearing about other cities like Kyoto and Osaka, but I think Kyoto used to be the capital a long time ago. So, maybe Tokyo took over as the capital more recently. I should probably double-check, but I think the answer is Tokyo. I can't think of any other city that's the capital. Yeah, I'm pretty confident it's Tokyo.
</think>

The capital of Japan is Tokyo.

FINAL: The capital of Japan is Tokyo.
Token limit hit: False
Seconds: 11.709


## 6. Define a small seven-category test suite
One question per category is only a smoke test, not a benchmark score. Review against the criteria below.
For coding: inspect output here; execute generated code only in a properly isolated sandbox.
For JSON: judge the extracted final answer without repairing it; optionally also evaluate raw-output compliance separately.


In [13]:
TESTS = [
    {
        "category": "Reasoning & Logic",
        "prompt": "All Nems are Zors. No Zors are Lats. Can any Nem be a Lat? Explain briefly and give a final yes/no answer.",
        "review_criteria": "Final answer: No. Explanation must follow the two premises.",
    },
    {
        "category": "Coding",
        "prompt": "Write a Python function unique_in_order(items) that removes duplicates while preserving first-occurrence order. Inputs are hashable. Include tests for an empty list and repeated values.",
        "review_criteria": "Correct Python; preserves order; handles empty/repeated input. Functional correctness requires sandboxed tests, not inspection alone.",
    },
    {
        "category": "Instruction Following",
        "prompt": "In your final answer, write exactly three bullet points about password safety. Each bullet must contain at most eight words. No heading or closing sentence.",
        "review_criteria": "Exactly three final-answer bullets; each at most eight whitespace-separated words; no extra prose.",
    },
    {
        "category": "Writing & Communication",
        "prompt": "Write a professional customer email under 80 words. Apologize that delivery is delayed, say the delivery date is not confirmed, and promise an update tomorrow. Do not invent a cause or delivery date.",
        "review_criteria": "Clear and polite; under 80 words; all requested facts; no invented cause/date.",
    },
    {
        "category": "Structured Output",
        "prompt": 'Your final answer must be only a valid JSON object, without markdown, containing exactly these fields: name (string) = Aye, age (integer) = 28, active (boolean) = true.',
        "review_criteria": 'json.loads succeeds; exact object {"name":"Aye","age":28,"active":true}; no extra text or fields in final answer.',
    },
    {
        "category": "Context Utilization",
        "prompt": "Use only this context: Product Orion costs 25 USD. Its warranty is 12 months. No delivery date is provided. Question: What is its price, warranty, and delivery date? If missing, say not provided.",
        "review_criteria": "25 USD; 12 months; delivery date not provided; no invented facts.",
    },
    {
        "category": "Multilingual / Burmese",
        "prompt": "ဖောက်သည်တစ်ယောက်ကို ပစ္စည်းပို့ဆောင်မှု နောက်ကျနေကြောင်း မြန်မာဘာသာနဲ့ ယဉ်ယဉ်ကျေးကျေး အသိပေးစာ ရေးပါ။ ပို့ဆောင်မယ့်ရက်ကို မသိသေးပါ။ မသိတဲ့ရက်နဲ့ အကြောင်းပြချက်ကို မဖန်တီးပါနဲ့။",
        "review_criteria": "Final answer is natural, polite Burmese; explains delay; no invented delivery date/cause. Review by a fluent Burmese speaker.",
    },
]
print("Test cases:", len(TESTS))


Test cases: 7


## ⬜ [UNCHANGED — မပြောင်းထားပါ] 7. Run and save each result
This deliberately starts a NEW run and overwrites the previous results file. Download prior results first if needed.
You may change MAX_NEW_TOKENS to 2048 if answers are truncated and memory permits, but record any setting changes.


In [14]:
MAX_NEW_TOKENS = 1024
SEED = 42
RESULTS_PATH = Path("/kaggle/working/deepseek_results.jsonl")
results = []
with RESULTS_PATH.open("w", encoding="utf-8") as saved:
    for test in TESTS:
        print("\n===", test["category"], "===", flush=True)
        try:
            result = run_prompt(test["prompt"], max_new_tokens=MAX_NEW_TOKENS, seed=SEED)
        except Exception as exc:
            print("Stopped:", type(exc).__name__, str(exc))
            print("Earlier completed results remain at", RESULTS_PATH)
            raise
        result.update(category=test["category"], review_criteria=test["review_criteria"], human_verdict="", human_notes="")
        results.append(result)
        saved.write(json.dumps(result, ensure_ascii=False) + "\n")
        saved.flush()
        print(result["final_answer"] if result["final_answer"] is not None else result["raw_output"])
        print("Seconds:", result["seconds"], "| token limit:", result["hit_token_limit"])
        print("Review:", test["review_criteria"])
assert len(results) == len(TESTS), "Not all tests completed"
print("\nCompleted", len(results), "cases. No automatic quality score has been assigned.")



=== Reasoning & Logic ===
No, no Nem can be a Lat. All Nems are Zors, and since no Zors are Lats, there is no overlap between Nems and Lats.

**Final Answer:** No, no Nem can be a Lat.
Seconds: 33.122 | token limit: False
Review: Final answer: No. Explanation must follow the two premises.

=== Coding ===
To solve this problem, we need to write a function that removes duplicates from a list while preserving the order of their first occurrence. The function should handle both empty lists and lists with repeated values efficiently.

### Approach
The approach to solve this problem involves using a set to keep track of seen items and a list to build the result. Here's a step-by-step breakdown of the approach:

1. **Initialize an empty list**: This will hold the final result where each item appears only once.
2. **Initialize an empty set**: This will keep track of items that have already been added to the result list.
3. **Iterate over each item in the input list**: For each item, check if 

## 🟥 [FIX — results မရှိသည့် error ပြင်ထားသည်] 7A0. Restore saved results

Run before 7A. Existing results stay in memory; otherwise read `/content/deepseek_results.jsonl`. No model inference is repeated. If the runtime was deleted, upload the JSONL extracted from your downloaded ZIP first. If no saved results exist, run step 7 after the model/setup prerequisites. Never use an empty results list as a workaround. Loading fewer than seven records indicates an incomplete original run.


In [15]:
# [FIX 7A0] Run this BEFORE 7A; no model loading or inference.
import json
from pathlib import Path

RESULTS_FOLDER = Path("/content")

if "results" not in globals() or results == []:
    saved_path = RESULTS_FOLDER / "deepseek_results.jsonl"
    if not saved_path.is_file():
        raise FileNotFoundError(
            "No results in this session, and deepseek_results.jsonl was not found. "
            "Upload your previously saved JSONL to Colab Files, or run step 7 "
            "after its setup/model prerequisites. Do not create empty/fake results."
        )
    loaded_results = [
        json.loads(line)
        for line in saved_path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip()
    ]
else:
    loaded_results = results

if not isinstance(loaded_results, list) or not loaded_results:
    raise ValueError("The saved results must be a non-empty list of real test records.")
if not all(
    isinstance(row, dict) and all(key in row for key in ["category", "prompt", "raw_output"])
    for row in loaded_results
):
    raise ValueError("Invalid results format: category, prompt, and raw_output are required.")

results = loaded_results
print("Results ready:", len(results), "cases. Model inference was NOT repeated.")
for index, row in enumerate(results, 1):
    print(f"\n{index}. {row['category']}")
    print(row.get("final_answer") or row["raw_output"])
    print("Review criteria:", row.get("review_criteria", "Refer to the original test criteria."))
print("\nNow fill 7A, then run 7B. If fewer than seven records were saved, step 7 did not finish.")


Results ready: 7 cases. Model inference was NOT repeated.

1. Reasoning & Logic
No, no Nem can be a Lat. All Nems are Zors, and since no Zors are Lats, there is no overlap between Nems and Lats.

**Final Answer:** No, no Nem can be a Lat.
Review criteria: Final answer: No. Explanation must follow the two premises.

2. Coding
To solve this problem, we need to write a function that removes duplicates from a list while preserving the order of their first occurrence. The function should handle both empty lists and lists with repeated values efficiently.

### Approach
The approach to solve this problem involves using a set to keep track of seen items and a list to build the result. Here's a step-by-step breakdown of the approach:

1. **Initialize an empty list**: This will hold the final result where each item appears only once.
2. **Initialize an empty set**: This will keep track of items that have already been added to the result list.
3. **Iterate over each item in the input list**: For 

## 🟩 [NEW — အသစ်ထည့်ထားသည်] 7A. Human review entry

**အဆင့် ၇ အောက်တွင် အသစ်ထည့်ထားသောအပိုင်း — အဖြေများကိုစစ်ပြီး PASS / FAIL / INCOMPLETE ဖြည့်ရန်။**
Keep step 7 outputs; do not rerun inference just to calculate percentages.
Fill each blank with PASS, FAIL, or INCOMPLETE based on its actual output and review criteria, then run this cell.
Use INCOMPLETE for an unfinished answer; a token-limit flag alone does not prove incorrectness.
Missing thinking boundaries require inspecting raw output. Do not automatically grade them FAIL.
For code, do not claim functional correctness until isolated tests have run.


In [16]:
# ===== [NEW 7A START] လူက review ဖြည့်သည့် cell အသစ် =====
# Review actual outputs before filling these values.
# PASS = meets criteria; FAIL = does not; INCOMPLETE = unfinished.
# Leave "" for not yet reviewed. These are not automatic grades.
verdicts = [
    "",  # 1. Reasoning & Logic
    "",  # 2. Coding (functional PASS needs sandboxed testing)
    "",  # 3. Instruction Following
    "",  # 4. Writing & Communication
    "",  # 5. Structured Output
    "",  # 6. Context Utilization
    "",  # 7. Multilingual / Burmese
]
if len(verdicts) != len(results):
    raise ValueError("This entry cell expects the original seven tests. Add one verdict per result if you changed the suite.")
if any(v.strip().upper() not in {"PASS", "FAIL", "INCOMPLETE", ""} for v in verdicts):
    raise ValueError("Use PASS, FAIL, INCOMPLETE, or an empty string.")
for row, verdict in zip(results, verdicts):
    row["human_verdict"] = verdict.strip().upper()
print("Reviews updated. Run 7B next; no model inference will be repeated.")

# ===== [NEW 7A END] =====


Reviews updated. Run 7B next; no model inference will be repeated.


## 🟩 [NEW — အသစ်ထည့်ထားသည်] 7B. Percentage report

**အသစ်ထည့်ထားသောအပိုင်း — category percentage၊ overall pass rate၊ review coverage နဲ့ summary ဖိုင်များ ထုတ်ပေးရန်။**
Run after 7A. Existing answers, timings, and raw-run JSONL remain available.
- **Pass rate (judged)** = PASS / (PASS + FAIL). N/A when nothing is judged.
- **Judged coverage** = (PASS + FAIL) / all completed cases.
- **Confirmed passes (all)** = PASS / all completed cases. This is not an assertion that pending/incomplete cases failed.
Counts for incomplete and pending cases are always shown; a partial pass rate must not be called overall model accuracy.
With one case per category, a judged category is only 0% or 100%. More representative questions are needed for useful percentages.


In [17]:
# ===== [NEW 7B START] Percentage report cell အသစ် =====
from collections import Counter
import csv
import json
from pathlib import Path


def summarize_reviews(rows):
    allowed = {"PASS", "FAIL", "INCOMPLETE", ""}
    counts = Counter()
    groups = {}
    for row in rows:
        verdict = row.get("human_verdict", "").strip().upper()
        if verdict not in allowed:
            raise ValueError(f"Invalid verdict: {verdict!r}")
        status = verdict or "PENDING"
        counts[status] += 1
        groups.setdefault(row["category"], Counter())[status] += 1

    def metrics(c):
        total = sum(c.values())
        judged = c["PASS"] + c["FAIL"]
        return {
            "total": total,
            "pass": c["PASS"], "fail": c["FAIL"],
            "incomplete": c["INCOMPLETE"], "pending": c["PENDING"],
            "pass_rate_judged_pct": round(100 * c["PASS"] / judged, 2) if judged else None,
            "judged_coverage_pct": round(100 * judged / total, 2) if total else None,
            "confirmed_pass_all_pct": round(100 * c["PASS"] / total, 2) if total else None,
        }
    return {"overall": metrics(counts), "categories": {k: metrics(v) for k, v in groups.items()}}


# Fill these only AFTER reviewing the actual outputs against review_criteria.
# Allowed: PASS, FAIL, INCOMPLETE; leave "" for not yet reviewed.
# Keep the same order as results. Existing verdicts are preserved by default.
verdicts = [row.get("human_verdict", "") for row in results]
# To edit, replace the line above, e.g. with a list of your own seven verdicts.
# Do not copy invented PASS/FAIL values from an example.

if len(verdicts) != len(results):
    raise ValueError("One verdict is required for each result.")
if any(v.strip().upper() not in {"PASS", "FAIL", "INCOMPLETE", ""} for v in verdicts):
    raise ValueError("Use only PASS, FAIL, INCOMPLETE, or an empty string.")
for row, verdict in zip(results, verdicts):
    row["human_verdict"] = verdict.strip().upper()

summary = summarize_reviews(results)

def pct(value):
    return "N/A" if value is None else f"{value:.2f}%"

print("\nCATEGORY RESULTS (human-reviewed)")
for category, m in summary["categories"].items():
    print(f"{category}: pass rate {pct(m['pass_rate_judged_pct'])} | "
          f"PASS {m['pass']}, FAIL {m['fail']}, INCOMPLETE {m['incomplete']}, PENDING {m['pending']}")
m = summary["overall"]
print("\nOverall pass rate (PASS / [PASS + FAIL]):", pct(m["pass_rate_judged_pct"]))
print("Judged coverage ([PASS + FAIL] / all):", pct(m["judged_coverage_pct"]))
print("Confirmed passes (PASS / all):", pct(m["confirmed_pass_all_pct"]))
print("Counts:", {k: m[k] for k in ["total", "pass", "fail", "incomplete", "pending"]})
print("One case per category gives only 0% or 100% when judged; this is not a benchmark score.")

# Separate files: do not overwrite the original raw-run JSONL.
folder = Path("/content")
summary_path = folder / "deepseek_percentage_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
reviewed_path = folder / "deepseek_reviewed_results.jsonl"
reviewed_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in results), encoding="utf-8")
summary_csv_path = folder / "deepseek_percentage_summary.csv"
summary_rows = [{"category": name, **values} for name, values in summary["categories"].items()]
summary_rows.append({"category": "OVERALL", **summary["overall"]})
with summary_csv_path.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)
print("Saved percentage summary and reviewed results. Run step 8 to download.")

# ===== [NEW 7B END] =====



CATEGORY RESULTS (human-reviewed)
Reasoning & Logic: pass rate N/A | PASS 0, FAIL 0, INCOMPLETE 0, PENDING 1
Coding: pass rate N/A | PASS 0, FAIL 0, INCOMPLETE 0, PENDING 1
Instruction Following: pass rate N/A | PASS 0, FAIL 0, INCOMPLETE 0, PENDING 1
Writing & Communication: pass rate N/A | PASS 0, FAIL 0, INCOMPLETE 0, PENDING 1
Structured Output: pass rate N/A | PASS 0, FAIL 0, INCOMPLETE 0, PENDING 1
Context Utilization: pass rate N/A | PASS 0, FAIL 0, INCOMPLETE 0, PENDING 1
Multilingual / Burmese: pass rate N/A | PASS 0, FAIL 0, INCOMPLETE 0, PENDING 1

Overall pass rate (PASS / [PASS + FAIL]): N/A
Judged coverage ([PASS + FAIL] / all): 0.00%
Confirmed passes (PASS / all): 0.00%
Counts: {'total': 7, 'pass': 0, 'fail': 0, 'incomplete': 0, 'pending': 7}
One case per category gives only 0% or 100% when judged; this is not a benchmark score.
Saved percentage summary and reviewed results. Run step 8 to download.


## 🟨 [UPDATED — ပြင်ဆင်ထားသည်] 8. Export results + environment

**မူလ export အပြင် percentage summary CSV / JSON နဲ့ reviewed results ဖိုင်များကို ZIP ထဲ ထည့်ပေးရန် ပြင်ဆင်ထားသည်။**
Download contains CSV for human review, original JSONL, environment metadata, and pip freeze.
Complete 7A and 7B before export to include the reviewed results and percentage summaries; record truncation separately and do not silently discard failures.
Expand to representative company questions (e.g. 20 per category as a pilot), repeat with multiple seeds, and test safety before production decisions.
Do not compare this small suite directly against published benchmark scores.


In [18]:
# [UPDATED 8] ZIP export (Kaggle version)
import subprocess
import zipfile
import json
import csv
import sys
from pathlib import Path

csv_path = Path("/kaggle/working/deepseek_results.csv")
with csv_path.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0]))
    writer.writeheader()
    writer.writerows(results)

metadata = {
    "model_id": MODEL_ID,
    "resolved_model_commit": getattr(model.config, "_commit_hash", None),
    "python": sys.version,
    "torch": torch.__version__,
    "cuda_build": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "packages": {p: version(p) for p in ["transformers", "accelerate", "bitsandbytes", "safetensors"]},
    "quantization": "NF4 4-bit, double quantization, FP16 compute",
    "attention": "sdpa",
    "temperature": 0.6,
    "top_p": 0.95,
    "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS,
    "completed_cases": len(results),
}

meta_path = Path("/kaggle/working/deepseek_environment.json")
meta_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

freeze_path = Path("/kaggle/working/deepseek_pip_freeze.txt")
freeze_path.write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")

archive = Path("/kaggle/working/deepseek_evaluation.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    extra_paths = [Path("/kaggle/working/deepseek_percentage_summary.json"), Path("/kaggle/working/deepseek_percentage_summary.csv"), Path("/kaggle/working/deepseek_reviewed_results.jsonl")]
    # RESULTS_PATH ကိုလည်း /kaggle/working/ ထဲမှာ ရှိတယ်လို့ ယူဆပြီး Path ပြင်ပေးပါ
    for path in [Path("/kaggle/working/deepseek_results.jsonl"), csv_path, meta_path, freeze_path] + [p for p in extra_paths if p.exists()]:
        z.write(path, arcname=path.name)

print("ZIP archive created successfully at:", archive)



ZIP archive created successfully at: /kaggle/working/deepseek_evaluation.zip


In [19]:
from pathlib import Path
print(Path("/kaggle/working/deepseek_results.jsonl").exists())

True


In [20]:
import os
from pathlib import Path

# /kaggle/working/ ထဲမှာရှိတဲ့ ဖိုင်များကို ရှာရန်
output_dir = Path("/kaggle/working/")
print("Files in working directory:")
for file_path in output_dir.glob("*"):
    print(file_path)

Files in working directory:
/kaggle/working/deepseek_environment.json
/kaggle/working/deepseek_pip_freeze.txt
/kaggle/working/.virtual_documents
/kaggle/working/deepseek_results.csv
/kaggle/working/deepseek_results.jsonl
/kaggle/working/deepseek_evaluation.zip


## 9. Visible reasoning and logic report

This summarizes only text the model generated and visible markers in that text. It cannot reveal hidden internal reasoning or verify that the explanation matches the model's actual computation. Logic steps and uncertainty/change markers are heuristic text cues; review the raw output before drawing conclusions.


In [21]:
import json
import re
from pathlib import Path


def split_think(text):
    text = text or ""
    if "<think>" in text and "</think>" in text:
        body = text.split("<think>", 1)[1].split("</think>", 1)[0].strip()
        state = "closed"
    elif "<think>" in text:
        body = text.split("<think>", 1)[1].strip()
        state = "open"
    else:
        body = ""
        state = "not_visible"
    return body, state


def reasoning_profile(row):
    raw = row.get("raw_output", "") or ""
    visible_reasoning, think_state = split_think(raw)
    lines = [line.strip() for line in visible_reasoning.splitlines() if line.strip()]
    numbered_or_bulleted = [
        re.sub(r"^(?:\d+[.)]|[-*•])\s+", "", line).strip()
        for line in lines
        if re.match(r"^(?:\d+[.)]|[-*•])\s+", line)
    ]
    logic_steps = numbered_or_bulleted[:10] if numbered_or_bulleted else lines[:5]

    lower_text = (raw + " " + visible_reasoning).lower()
    change_cues = [
        cue for cue in ["wait", "actually", "on second thought", "let me reconsider", "correction", "ပြန်စဉ်းစား", "ပြန်လည်စဉ်းစား"]
        if cue in lower_text
    ]
    uncertainty_cues = [
        cue for cue in ["maybe", "likely", "probably", "depends", "could", "might", "unclear", "uncertain", "ဖြစ်နိုင်", "မသေချာ", "ထင်ပါတယ်", "ယူဆရ"]
        if cue in lower_text
    ]
    return {
        "category": row.get("category", ""),
        "think_state": think_state,
        "visible_reasoning": visible_reasoning,
        "logic_steps_heuristic": logic_steps,
        "change_of_thought_cues_heuristic": change_cues,
        "soft_logic_cues_heuristic": uncertainty_cues,
        "interpretation_limit": "Text-based cues only; not a measurement of hidden reasoning or correctness.",
        "final_answer": row.get("final_answer", ""),
    }


if "results" not in globals() or not results:
    source_path = Path("/kaggle/working/deepseek_results.jsonl")
    if not source_path.is_file():
        raise FileNotFoundError("Run step 7 or upload deepseek_results.jsonl before creating a reasoning summary.")
    results = [
        json.loads(line)
        for line in source_path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip()
    ]

profiles = [reasoning_profile(row) for row in results]
reasoning_path = Path("/content/deepseek_reasoning_analysis.jsonl")
reasoning_path.write_text(
    "".join(json.dumps(profile, ensure_ascii=False) + "\n" for profile in profiles),
    encoding="utf-8",
)
print("Saved visible-text reasoning summaries:", len(profiles), "rows to", reasoning_path)


Saved visible-text reasoning summaries: 7 rows to /content/deepseek_reasoning_analysis.jsonl


## 10. Token probabilities

This helper performs fresh sampled generation and reports per-token probability, entropy, and top-1/top-2 margin from Transformers generation scores. With temperature and top-p enabled, these describe the filtered sampling distribution—not raw unfiltered model probabilities, answer correctness, or calibrated confidence. Earlier saved outputs do not contain these scores, so they cannot be recovered without rerunning inference.


In [22]:
import time
import torch
import torch.nn.functional as F


def run_with_scores(prompt, max_new_tokens=64, seed=42):
    """Generate a fresh answer and summarize post-sampling token scores."""
    set_seed(seed)
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    if not rendered.rstrip().endswith("<think>"):
        rendered += "<think>\n"
    inputs = tokenizer(
        rendered,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda:0")
    input_tokens = inputs["input_ids"].shape[-1]
    if input_tokens > 2048:
        raise ValueError("This report limits inputs to 2048 tokens; shorten the prompt.")

    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        generated_output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=model.generation_config.eos_token_id,
            return_dict_in_generate=True,
            output_scores=True,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    generated_ids = generated_output.sequences[0, input_tokens:]
    token_rows = []
    for index, (step_scores, token_id) in enumerate(
        zip(generated_output.scores, generated_ids.tolist()), start=1
    ):
        # These scores reflect temperature/top-p sampling, not calibrated correctness.
        distribution = F.softmax(step_scores[0].float(), dim=-1)
        chosen_probability = distribution[token_id].clamp_min(1e-12)
        top_two = torch.topk(distribution, k=2).values
        entropy = -(distribution * distribution.clamp_min(1e-12).log()).sum()
        token_rows.append({
            "token_index": index,
            "token": tokenizer.decode([token_id], skip_special_tokens=False),
            "chosen_token_probability": float(chosen_probability.item()),
            "chosen_token_log_probability": float(chosen_probability.log().item()),
            "sampling_entropy": float(entropy.item()),
            "top1_top2_probability_margin": float((top_two[0] - top_two[1]).item()),
        })

    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=False)
    final_answer, extraction = extract_final(raw_output)
    count = len(token_rows)
    if count:
        mean_probability = sum(row["chosen_token_probability"] for row in token_rows) / count
        mean_log_probability = sum(row["chosen_token_log_probability"] for row in token_rows) / count
        mean_entropy = sum(row["sampling_entropy"] for row in token_rows) / count
        mean_margin = sum(row["top1_top2_probability_margin"] for row in token_rows) / count
        geometric_mean_probability = float(torch.tensor(mean_log_probability).exp().item())
    else:
        mean_probability = mean_log_probability = None
        mean_entropy = mean_margin = geometric_mean_probability = None

    return {
        "prompt": prompt,
        "raw_output": raw_output,
        "final_answer": final_answer,
        "answer_extraction": extraction,
        "input_tokens": input_tokens,
        "output_tokens": count,
        "seconds": round(elapsed, 3),
        "seed": seed,
        "max_new_tokens": max_new_tokens,
        "sampling": {"do_sample": True, "temperature": 0.6, "top_p": 0.95},
        "probability_scope": "Sampling-distribution probabilities after temperature and top-p; not answer correctness confidence.",
        "average_chosen_token_probability": mean_probability,
        "geometric_mean_chosen_token_probability": geometric_mean_probability,
        "average_chosen_token_log_probability": mean_log_probability,
        "average_sampling_entropy": mean_entropy,
        "average_top1_top2_probability_margin": mean_margin,
        "token_details": token_rows,
    }


print("Score helper ready. It measures a fresh sampled response; it does not recover scores from earlier runs.")

Score helper ready. It measures a fresh sampled response; it does not recover scores from earlier runs.


## 11. Generate and view the reasoning + probability report

Run this after the model is loaded and the score helper in section 10 has run. It performs a fresh sampled inference for each test, displays a compact report in the notebook, and saves detailed JSONL plus CSV and ZIP downloads under `/content`. It does not overwrite the original results JSONL. The report's token probabilities are sampling-distribution scores, not confidence that an answer is correct. The default 64-token report limit is intentionally smaller to keep score memory use manageable; increase only if GPU memory permits.


In [25]:
import csv
import json
import zipfile
from pathlib import Path
from IPython.display import Markdown, display
from google.colab import files as colab_files

REPORT_MAX_NEW_TOKENS = 64
REPORT_SEED = 42
report_jsonl_path = Path("/kaggle/working/deepseek_reasoning_probability_report.jsonl")
report_csv_path = Path("/kaggle/working/deepseek_reasoning_probability_summary.csv")
report_zip_path = Path("/kaggle/working/deepseek_reasoning_probability_report.zip")
report_records = []

if "TESTS" not in globals() or not TESTS:
    raise ValueError("TESTS is empty or missing. Run the test-suite cell first.")

# Write each completed case immediately so partial progress survives an interruption.
with report_jsonl_path.open("w", encoding="utf-8") as report_file:
    for index, test in enumerate(TESTS, start=1):
        print(f"Scoring report case {index}/{len(TESTS)}: {test['category']}", flush=True)
        scored = run_with_scores(
            test["prompt"],
            max_new_tokens=REPORT_MAX_NEW_TOKENS,
            seed=REPORT_SEED,
        )
        scored["category"] = test["category"]
        scored["review_criteria"] = test.get("review_criteria", "")
        scored["report_run"] = "fresh_sampled_generation"
        profile = reasoning_profile(scored)
        scored["visible_reasoning_report"] = profile
        report_records.append(scored)
        report_file.write(json.dumps(scored, ensure_ascii=False) + "\n")
        report_file.flush()

summary_rows = []
for record in report_records:
    profile = record["visible_reasoning_report"]
    summary_rows.append({
        "category": record["category"],
        "prompt": record["prompt"],
        "think_state": profile["think_state"],
        "logic_steps_heuristic": " | ".join(profile["logic_steps_heuristic"]),
        "change_of_thought_cues_heuristic": ", ".join(profile["change_of_thought_cues_heuristic"]),
        "soft_logic_cues_heuristic": ", ".join(profile["soft_logic_cues_heuristic"]),
        "output_tokens": record["output_tokens"],
        "average_chosen_token_probability": record["average_chosen_token_probability"],
        "geometric_mean_chosen_token_probability": record["geometric_mean_chosen_token_probability"],
        "average_sampling_entropy": record["average_sampling_entropy"],
        "average_top1_top2_probability_margin": record["average_top1_top2_probability_margin"],
        "final_answer": record["final_answer"],
        "human_review_required": True,
    })

with report_csv_path.open("w", encoding="utf-8-sig", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)


def report_text(value, limit=180):
    text = str(value or "").replace("|", "\\|").replace("\n", "<br>")
    return text if len(text) <= limit else text[:limit - 1] + "…"


def report_metric(value):
    return "N/A" if value is None else f"{value:.4f}"


table_lines = [
    "| Category | Think text | Logic cues (heuristic) | Change cues | Soft/uncertainty cues | Avg token probability | Geometric mean | Avg entropy | Top1-top2 margin | Final answer |",
    "|---|---|---|---|---|---:|---:|---:|---:|---|",
]
for record in report_records:
    profile = record["visible_reasoning_report"]
    table_lines.append(
        "| " + " | ".join([
            report_text(record["category"], 45),
            report_text(profile["think_state"], 24),
            report_text("; ".join(profile["logic_steps_heuristic"]), 100),
            report_text(", ".join(profile["change_of_thought_cues_heuristic"]), 60),
            report_text(", ".join(profile["soft_logic_cues_heuristic"]), 60),
            report_metric(record["average_chosen_token_probability"]),
            report_metric(record["geometric_mean_chosen_token_probability"]),
            report_metric(record["average_sampling_entropy"]),
            report_metric(record["average_top1_top2_probability_margin"]),
            report_text(record["final_answer"], 120),
        ]) + " |"
    )

display(Markdown("\n".join([
    "### Reasoning + token probability report",
    "Fresh sampled responses; probability metrics describe the temperature/top-p sampling distribution, not answer correctness.",
    "Logic/change/uncertainty cues are text heuristics. Inspect each raw output and review criteria before grading.",
    "",
    *table_lines,
    "",
    f"Detailed token-by-token JSONL: `{report_jsonl_path.name}`  ",
    f"Summary CSV: `{report_csv_path.name}`",
])))

with zipfile.ZipFile(report_zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(report_jsonl_path, arcname=report_jsonl_path.name)
    archive.write(report_csv_path, arcname=report_csv_path.name)
print("Saved report files:", report_jsonl_path, report_csv_path, report_zip_path)
#colab_files.download(str(report_zip_path))
print("Files saved successfully!")

Scoring report case 1/7: Reasoning & Logic
Scoring report case 2/7: Coding
Scoring report case 3/7: Instruction Following
Scoring report case 4/7: Writing & Communication
Scoring report case 5/7: Structured Output
Scoring report case 6/7: Context Utilization
Scoring report case 7/7: Multilingual / Burmese


### Reasoning + token probability report
Fresh sampled responses; probability metrics describe the temperature/top-p sampling distribution, not answer correctness.
Logic/change/uncertainty cues are text heuristics. Inspect each raw output and review criteria before grading.

| Category | Think text | Logic cues (heuristic) | Change cues | Soft/uncertainty cues | Avg token probability | Geometric mean | Avg entropy | Top1-top2 margin | Final answer |
|---|---|---|---|---|---:|---:|---:|---:|---|
| Reasoning & Logic | not_visible |  |  |  | 0.9193 | 0.8966 | 0.1505 | 0.8596 |  |
| Coding | not_visible |  |  |  | 0.8947 | 0.8348 | 0.1632 | 0.8655 |  |
| Instruction Following | not_visible |  |  |  | 0.8429 | 0.7436 | 0.2250 | 0.8145 |  |
| Writing & Communication | not_visible |  |  |  | 0.8494 | 0.7730 | 0.2245 | 0.8016 |  |
| Structured Output | not_visible |  |  |  | 0.9216 | 0.9019 | 0.1694 | 0.8693 |  |
| Context Utilization | not_visible |  |  |  | 0.8824 | 0.8106 | 0.1752 | 0.8594 |  |
| Multilingual / Burmese | not_visible |  |  | maybe | 0.7905 | 0.7007 | 0.4643 | 0.7300 |  |

Detailed token-by-token JSONL: `deepseek_reasoning_probability_report.jsonl`  
Summary CSV: `deepseek_reasoning_probability_summary.csv`

Saved report files: /kaggle/working/deepseek_reasoning_probability_report.jsonl /kaggle/working/deepseek_reasoning_probability_summary.csv /kaggle/working/deepseek_reasoning_probability_report.zip
Files saved successfully!


## 12. Inspect the visible reasoning and raw outputs

This cell does not call the model again. It expands each report case so the generated visible reasoning, heuristic logic/change/uncertainty cues, final answer, and sampling-score metrics can be inspected. If the report cell ran in an earlier session, it reloads the separate report JSONL.


In [26]:
import html
import json
from pathlib import Path
from IPython.display import HTML, display

if "report_records" not in globals() or not report_records:
    report_jsonl_path = Path("/kaggle/working/deepseek_reasoning_probability_report.jsonl")
    if not report_jsonl_path.is_file():
        raise FileNotFoundError("Run section 11 first, or upload its reasoning_probability_report.jsonl file.")
    report_records = [
        json.loads(line)
        for line in report_jsonl_path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip()
    ]

html_sections = []
for index, record in enumerate(report_records, start=1):
    profile = record.get("visible_reasoning_report") or reasoning_profile(record)
    category = html.escape(str(record.get("category", "")))
    visible_reasoning = html.escape(profile.get("visible_reasoning") or "(No visible <think> text was found.)")
    final_answer = html.escape(str(record.get("final_answer") or "(No final answer extracted.)"))
    logic_cues = html.escape("\n".join(profile.get("logic_steps_heuristic", [])) or "(No numbered/bulleted logic cues found.)")
    change_cues = html.escape(", ".join(profile.get("change_of_thought_cues_heuristic", [])) or "None detected")
    soft_cues = html.escape(", ".join(profile.get("soft_logic_cues_heuristic", [])) or "None detected")
    probability = record.get("average_chosen_token_probability")
    probability_text = "N/A" if probability is None else f"{probability:.6f}"
    entropy = record.get("average_sampling_entropy")
    entropy_text = "N/A" if entropy is None else f"{entropy:.6f}"
    html_sections.append(
        f"<details><summary><strong>{index}. {category}</strong> — "
        f"avg sampled-token probability: {probability_text}</summary>"
        f"<p><strong>Visible reasoning:</strong></p><pre style='white-space:pre-wrap'>{visible_reasoning}</pre>"
        f"<p><strong>Heuristic logic cues:</strong></p><pre style='white-space:pre-wrap'>{logic_cues}</pre>"
        f"<p><strong>Change cues:</strong> {change_cues}<br>"
        f"<strong>Soft/uncertainty cues:</strong> {soft_cues}<br>"
        f"<strong>Average sampling entropy:</strong> {entropy_text}</p>"
        f"<p><strong>Final answer:</strong></p><pre style='white-space:pre-wrap'>{final_answer}</pre>"
        f"</details>"
    )

display(HTML("\n".join([
    "<p>The earlier summary table's think-state is only a marker status. Full visible text is shown below.</p>",
    "<p>These are generated text and sampling-distribution measurements; they are not hidden reasoning or calibrated correctness confidence.</p>",
    *html_sections,
])))

## 13. Review saved results and report files

This optional cell reads the original run, human-review summaries, visible-reasoning analysis, and the fresh probability report if those files exist. It does not rerun inference or overwrite the earlier results.


In [27]:
import json
from pathlib import Path
from statistics import mean


def load_jsonl(path):
    path = Path(path)
    if not path.is_file():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip()
    ]


results_path = Path("/kaggle/working/deepseek_results.jsonl")
reviewed_path = Path("/kaggle/working/deepseek_reviewed_results.jsonl")
reasoning_path = Path("/kaggle/working/deepseek_reasoning_analysis.jsonl")
probability_path = Path("/kaggle/working/deepseek_reasoning_probability_report.jsonl")
percentage_path = Path("/kaggle/working/deepseek_percentage_summary.json")

raw_results = load_jsonl(results_path)
reviewed_results = load_jsonl(reviewed_path) or raw_results
reasoning_rows = load_jsonl(reasoning_path)
probability_rows = load_jsonl(probability_path)
percentage_summary = (
    json.loads(percentage_path.read_text(encoding="utf-8"))
    if percentage_path.is_file()
    else None
)

print("=== DeepSeek T4 evaluation report ===")
print("Original results:", len(raw_results))
print("Human-reviewed results:", len(reviewed_results))
print("Visible-reasoning rows:", len(reasoning_rows))
print("Fresh probability-report rows:", len(probability_rows))

if percentage_summary:
    overall = percentage_summary.get("overall", {})
    print("--- Human-review percentages ---")
    print("Judged pass rate (%):", overall.get("pass_rate_judged_pct"))
    print("Judged coverage (%):", overall.get("judged_coverage_pct"))
    print("Confirmed passes / all (%):", overall.get("confirmed_pass_all_pct"))

if reviewed_results:
    print("--- Reviewed answers ---")
    for index, row in enumerate(reviewed_results, start=1):
        print(
            f"{index}. {row.get('category', '')} | "
            f"verdict={row.get('human_verdict', 'PENDING')} | "
            f"answer={row.get('final_answer') or '(not extracted)'}"
        )

if reasoning_rows:
    print("--- Visible reasoning cues (heuristic) ---")
    for row in reasoning_rows:
        print(
            f"{row.get('category', '')}: "
            f"think_state={row.get('think_state', '')} | "
            f"soft cues={row.get('soft_logic_cues_heuristic', [])} | "
            f"change cues={row.get('change_of_thought_cues_heuristic', [])} | "
            f"logic cues={len(row.get('logic_steps_heuristic', []))}"
        )

if probability_rows:
    print("--- Fresh token-probability report ---")
    for row in probability_rows:
        print(
            f"{row.get('category', '')}: "
            f"avg sampled-token probability={row.get('average_chosen_token_probability')} | "
            f"avg entropy={row.get('average_sampling_entropy')} | "
            f"answer={row.get('final_answer') or '(not extracted)'}"
        )
    seconds = [row.get("seconds") for row in probability_rows if isinstance(row.get("seconds"), (int, float))]
    if seconds:
        print("Average report inference time (seconds):", round(mean(seconds), 3))

print("Report preview complete. Probability values are sampling scores, not correctness confidence.")

=== DeepSeek T4 evaluation report ===
Original results: 7
Human-reviewed results: 7
Visible-reasoning rows: 0
Fresh probability-report rows: 7
--- Reviewed answers ---
1. Reasoning & Logic | verdict= | answer=No, no Nem can be a Lat. All Nems are Zors, and since no Zors are Lats, there is no overlap between Nems and Lats.

**Final Answer:** No, no Nem can be a Lat.
2. Coding | verdict= | answer=To solve this problem, we need to write a function that removes duplicates from a list while preserving the order of their first occurrence. The function should handle both empty lists and lists with repeated values efficiently.

### Approach
The approach to solve this problem involves using a set to keep track of seen items and a list to build the result. Here's a step-by-step breakdown of the approach:

1. **Initialize an empty list**: This will hold the final result where each item appears only once.
2. **Initialize an empty set**: This will keep track of items that have already been added to

## 14. README

- Run cells in order.
- Keep the earlier results files unchanged.
- Use 11/13/14 only for viewing saved outputs and reports.
- Step 14 is a short reference section, not a rerun step.


## Troubleshooting
- **No CUDA GPU**: select a GPU runtime; do not run model loading on CPU by accident.
- **No matching distribution for an old torch**: keep bundled torch; do not reuse the old downgrade commands.
- **401 with a missing organization prefix**: use the exact MODEL_ID above.
- **Unauthenticated Hub warning**: not fatal by itself; optional Colab HF_TOKEN secret can increase limits.
- **CUDA / bitsandbytes import or kernel error**: stop and share cell 2 output plus full traceback. A configuration object alone does not test kernels.
- **Out of memory**: restart session to release live tensors; run from cell 2, load only one model; reduce prompt/output lengths. empty_cache cannot free tensors still referenced.
- **hit_token_limit=True**: output may be incomplete; increase budget if feasible and record rerun separately.

Sources: [model card](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B), [quantization](https://huggingface.co/docs/transformers/quantization/bitsandbytes), [bitsandbytes installation](https://huggingface.co/docs/bitsandbytes/installation).
